# Data Collection for Portfolio Optimization

## Overview

This notebook demonstrates the data collection process for portfolio optimization analysis. We collect historical stock price data for a diversified set of 10 assets using the Yahoo Finance API through the `yfinance` library.

### Objectives:
- Download daily price data for selected securities
- Calculate daily returns and compute annualized statistics
- Prepare data for statistical analysis and optimization

### Asset Universe:
Our portfolio includes 10 assets across different sectors:
- **Technology**: AAPL, MSFT, NVDA
- **Financial Services**: JPM, GS
- **Energy**: XOM
- **Healthcare**: JNJ
- **Consumer Discretionary**: AMZN
- **Consumer Staples**: PG
- **Commodities**: GLD (Gold ETF)

## Data Acquisition Methodology

We use Yahoo Finance as our data source due to its comprehensive coverage and free API access. The `yfinance` library provides a convenient Python interface for downloading historical market data.

### Data Parameters:
- **Period**: 3 months (recent market conditions)
- **Interval**: Daily prices
- **Price Type**: Close prices (adjusted for splits and dividends where available)

### Processing Steps:
1. Download raw price data for each ticker
2. Calculate daily percentage returns
3. Compute annualized mean returns and volatilities
4. Export data for analysis

In [ ]:
# Import required libraries
import yfinance as yf  # Yahoo Finance API wrapper
import pandas as pd    # Data manipulation and analysis

In [ ]:
# Define the list of tickers for our portfolio
tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GS', 'XOM', 'JNJ', 'AMZN', 'PG', 'GLD']

# Initialize dictionary to store computed statistics
df = {}

# Loop through each ticker to download data and compute statistics
for i in tickers:
    # Download historical data from Yahoo Finance
    data = yf.download(i, period='3mo', interval='1d')  # 3-month daily data

    # Extract closing prices (using 'Close' as proxy for adjusted close)
    adj = data["Close"]  # Note: For precision, 'Adj Close' would be better

    # Calculate daily percentage returns and remove NaN values
    returns = adj.pct_change().dropna().values

    # Compute daily statistics
    mean_daily_return = returns.mean()
    std_dev = returns.std()

    # Annualize the statistics (252 trading days in a year)
    mean_annual_return = mean_daily_return * 252
    std_annualized_dev = std_dev * (252 ** 0.5)

    # Store results in dictionary
    df[i] = {"Return": mean_annual_return, "STD": std_annualized_dev}

# Convert to DataFrame for better visualization
df1 = pd.DataFrame(df)
print("Summary Statistics (Annualized):")
print(df1)

# Save summary data to CSV
df1.to_csv("data.csv")

## Improved Implementation: Full Returns Data

The previous approach only computed summary statistics. For portfolio optimization, we need the full covariance matrix, which requires the complete time series of returns for all assets.

This implementation:
- Collects daily returns for all tickers in a single DataFrame
- Enables proper covariance matrix calculation
- Provides data for correlation analysis and optimization

In [ ]:
# Define portfolio tickers
tickers = ['AAPL', 'MSFT', 'NVDA', 'JPM', 'GS', 'XOM', 'JNJ', 'AMZN', 'PG', 'GLD']

# Initialize DataFrame to store returns for all tickers
all_returns = pd.DataFrame()

# Download data and compute returns for each ticker
for ticker in tickers:
    # Fetch historical price data
    data = yf.download(ticker, period='3mo', interval='1d')

    # Extract closing prices
    adj_close = data["Close"]

    # Calculate daily percentage returns and handle missing data
    daily_returns = adj_close.pct_change().dropna()

    # Add to the collective returns DataFrame
    all_returns[ticker] = daily_returns

# Save the complete returns dataset for analysis
all_returns.to_csv("returns_data.csv")

# Compute and save summary statistics (optional)
summary = pd.DataFrame({
    'Return': all_returns.mean() * 252,  # Annualized mean return
    'STD': all_returns.std() * (252 ** 0.5)  # Annualized standard deviation
}).T
summary.to_csv("summary_data.csv")

print("Data collection complete. Files saved:")
print("- returns_data.csv: Full daily returns matrix")
print("- summary_data.csv: Annualized statistics")